# 0 — Build the Week 1 benchmark corpus

This notebook is **one-time setup**, not a teaching segment. Run it once and notebooks 1-5 will load the cached artifacts in two lines.

We build a **deliberately heterogeneous** corpus from three sources:

1. **Beehiiv blog posts** — Sinan's [AI Office Hours](https://ai-office-hours.beehiiv.com/) (mid-length opinion / explainer posts).
2. **Wikipedia AI-history slice** — long, formal, fact-dense articles on a coherent theme (AI history, key models, key researchers).
3. **HotpotQA mini-slice** — short, factual paragraphs with explicit gold supporting facts (free supervision for the gold set).

All three go into a **single Chroma collection** with source metadata, plus a parallel BM25 index. This is what makes hybrid retrieval and reranking actually earn their keep in the third notebook, and what makes the bake-off in the fifth notebook a real test rather than a contrived one.

We also generate the **gold set** here: ~10 hand-curated cross-source multi-hop questions, ~10 single-source controls, and ~10 questions sampled from HotpotQA with their gold supporting facts.

Outputs (gitignored):
- `data/chroma_db/` — persistent Chroma index
- `data/bm25_index.pkl` — pickled BM25 + parallel docs
- `data/gold_set.jsonl` — ~30 question gold set
- `data/corpus_cache/` — raw scraped text per source

This notebook is **idempotent** — re-running is a no-op if the artifacts exist.

In [1]:
import json
import pickle
import random
from pathlib import Path

from tqdm import tqdm

from corpus import (
    CACHE_DIR,
    CHROMA_COLLECTION,
    CHROMA_DIR,
    DATA_DIR,
    GoldQuestion,
    Source,
    corpus_exists,
    get_embeddings,
    save_bm25,
    write_gold_set,
)
from retrievers import build_chroma_index

DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

REBUILD = False

if corpus_exists() and not REBUILD:
    print("Corpus artifacts already exist. Set REBUILD = True to force a rebuild.")
else:
    print("Building corpus from scratch...")

Building corpus from scratch...


## Step 1 — Beehiiv (AI Office Hours)

Hand-picked URLs from Sinan's [AI Office Hours](https://ai-office-hours.beehiiv.com/) Beehiiv. These posts cover RAG, evaluation, agents, quantization — exactly the topics learners will ask about. We use LangChain's `WebBaseLoader` and cache the cleaned text per URL so reruns are free.

In [2]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document

BEEHIIV_URLS = [
    "https://ai-office-hours.beehiiv.com/p/evaluating-ai-agent-tool-selection",
    "https://ai-office-hours.beehiiv.com/p/beyond-benchmarks",
    "https://ai-office-hours.beehiiv.com/p/re-ranking-rag",
    "https://ai-office-hours.beehiiv.com/p/quantizing-llms-llama-3",
    "https://ai-office-hours.beehiiv.com/p/llm-probing",
    "https://ai-office-hours.beehiiv.com/p/ais-supervising-ais",
]

BEEHIIV_DIR = CACHE_DIR / "beehiiv"
BEEHIIV_DIR.mkdir(parents=True, exist_ok=True)

# Beehiiv silently 302-redirects unknown slugs to a "404 page doesn't exist" template
# rendered with status 200, so we have to detect it from the body itself.
_BEEHIIV_404_FINGERPRINTS = (
    "The page you requested doesn't exist",
    "We could not locate the page",
)
_BEEHIIV_MIN_CHARS = 800  # real posts are 3k+ chars; the 404 template is ~2k.


def _looks_like_404(content: str) -> bool:
    return any(fp in content for fp in _BEEHIIV_404_FINGERPRINTS)


def load_beehiiv() -> list[Document]:
    docs: list[Document] = []
    for url in tqdm(BEEHIIV_URLS, desc="beehiiv"):
        slug = url.rsplit("/", 1)[-1]
        cache_file = BEEHIIV_DIR / f"{slug}.json"
        if cache_file.exists():
            payload = json.loads(cache_file.read_text())
        else:
            try:
                loaded = WebBaseLoader(url).load()
            except Exception as exc:
                raise RuntimeError(f"failed to load {url}: {exc}") from exc
            if not loaded:
                raise RuntimeError(f"empty response from {url}")
            d = loaded[0]
            payload = {"content": d.page_content, "title": d.metadata.get("title", slug), "url": url}
            if _looks_like_404(payload["content"]) or len(payload["content"]) < _BEEHIIV_MIN_CHARS:
                raise RuntimeError(
                    f"{url} returned a 404 / placeholder page "
                    f"({len(payload['content'])} chars). Check the slug on "
                    "https://ai-office-hours.beehiiv.com/"
                )
            cache_file.write_text(json.dumps(payload))
        # Defensive: if a previous (broken) run cached a 404, refuse to use it.
        if _looks_like_404(payload["content"]):
            cache_file.unlink(missing_ok=True)
            raise RuntimeError(
                f"cached content for {slug} is a 404 page; deleted cache, please re-run."
            )
        docs.append(
            Document(
                page_content=payload["content"],
                metadata={
                    "source": Source.BEEHIIV.value,
                    "title": payload["title"],
                    "url": payload["url"],
                    "doc_id": f"beehiiv::{slug}",
                },
            )
        )
    return docs


beehiiv_docs = load_beehiiv()
print(f"Loaded {len(beehiiv_docs)} Beehiiv posts (avg {sum(len(d.page_content) for d in beehiiv_docs)//max(len(beehiiv_docs),1)} chars)")

/Users/sinanozdemir/Teaching/Pearson/advanced-agentic-ai-in-three-weeks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.
beehiiv: 100%|██████████| 6/6 [00:00<00:00, 1954.17it/s]

Loaded 6 Beehiiv posts (avg 8603 chars)


## Step 2 — Wikipedia AI history slice

A curated set of articles on AI history, foundational architectures, and key researchers. Long, formal, fact-dense — a deliberate stylistic contrast to the Beehiiv posts.

We hit Wikipedia's MediaWiki API directly with `requests` (the popular `wikipedia` PyPI package broke when Wikipedia tightened its API requirements — it sends a default user-agent that gets refused). Wikipedia requires a descriptive `User-Agent` per their [API etiquette](https://meta.wikimedia.org/wiki/User-Agent_policy), so we set one. Pages are cached locally as JSON to make reruns instant.

In [3]:
import time
import requests

WIKI_TITLES = [
    "History of artificial intelligence",
    "Artificial intelligence",
    "Machine learning",
    "Deep learning",
    "Neural network (machine learning)",
    "Backpropagation",
    "Transformer (deep learning architecture)",
    "Attention (machine learning)",
    "Recurrent neural network",
    "Convolutional neural network",
    "Reinforcement learning",
    "Reinforcement learning from human feedback",
    "Large language model",
    "Generative pre-trained transformer",
    "GPT-3",
    "GPT-4",
    "BERT (language model)",
    "Word2vec",
    "Word embedding",
    "Retrieval-augmented generation",
    "Vector database",
    "Information retrieval",
    "Tf\u2013idf",
    "Okapi BM25",
    "Sentence embedding",
    "Knowledge graph",
    "AlexNet",
    "ImageNet",
    "AlphaGo",
    "AlphaFold",
    "DeepMind",
    "OpenAI",
    "Anthropic",
    "Hugging Face",
    "Geoffrey Hinton",
    "Yann LeCun",
    "Yoshua Bengio",
    "Andrew Ng",
    "Ilya Sutskever",
    "Demis Hassabis",
    "Alan Turing",
    "John McCarthy (computer scientist)",
    "Marvin Minsky",
    "Frank Rosenblatt",
    "Dartmouth workshop",
    "Turing test",
    "AI winter",
    "Expert system",
    "Symbolic artificial intelligence",
]

WIKI_DIR = CACHE_DIR / "wikipedia"
WIKI_DIR.mkdir(parents=True, exist_ok=True)

WIKI_API = "https://en.wikipedia.org/w/api.php"
# Wikipedia requires a descriptive User-Agent identifying the application + contact.
# Default UAs (and the `wikipedia` PyPI package's UA) are now refused.
WIKI_HEADERS = {
    "User-Agent": (
        "advanced-agentic-ai-course/1.0 "
        "(educational; https://github.com/sinanuozdemir/advanced-agentic-ai-in-three-weeks)"
    )
}


def _fetch_wiki_page(title: str, retries: int = 3) -> dict | None:
    """Fetch a Wikipedia page via the MediaWiki action API. Returns None on miss."""
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts|info",
        "explaintext": "1",
        "redirects": "1",
        "inprop": "url",
        "titles": title,
    }
    last_exc = None
    for attempt in range(retries):
        try:
            r = requests.get(WIKI_API, params=params, headers=WIKI_HEADERS, timeout=30)
            r.raise_for_status()
            data = r.json()
            pages = data.get("query", {}).get("pages", {})
            if not pages:
                return None
            _pid, page = next(iter(pages.items()))
            if page.get("missing") is not None or page.get("pageid") is None:
                return None
            extract = page.get("extract", "")
            if not extract.strip():
                return None
            return {
                "content": extract,
                "title": page.get("title", title),
                "url": page.get("fullurl", f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"),
            }
        except Exception as exc:
            last_exc = exc
            time.sleep(0.5 * (2**attempt))
    raise RuntimeError(f"wiki fetch failed after {retries} retries: {last_exc}")


def load_wikipedia() -> list[Document]:
    docs: list[Document] = []
    misses: list[str] = []
    for title in tqdm(WIKI_TITLES, desc="wikipedia"):
        slug = title.lower().replace(" ", "_").replace("/", "_")
        cache_file = WIKI_DIR / f"{slug}.json"
        if cache_file.exists():
            payload = json.loads(cache_file.read_text())
        else:
            try:
                payload = _fetch_wiki_page(title)
            except Exception as exc:
                print(f"  ! error  {title}: {exc}")
                continue
            if payload is None:
                misses.append(title)
                print(f"  ! miss   {title}: no Wikipedia page found")
                continue
            cache_file.write_text(json.dumps(payload))
            time.sleep(0.1)  # be polite to the API
        docs.append(
            Document(
                page_content=payload["content"],
                metadata={
                    "source": Source.WIKIPEDIA.value,
                    "title": payload["title"],
                    "url": payload["url"],
                    "doc_id": f"wikipedia::{slug}",
                },
            )
        )
    if misses:
        print(f"\n  Skipped {len(misses)} missing pages: {misses}")
    return docs


wiki_docs = load_wikipedia()
print(f"\nLoaded {len(wiki_docs)} Wikipedia articles (avg {sum(len(d.page_content) for d in wiki_docs)//max(len(wiki_docs),1)} chars)")

wikipedia: 100%|██████████| 49/49 [00:00<00:00, 3148.06it/s]


Loaded 49 Wikipedia articles (avg 33615 chars)


## Step 3 — HotpotQA mini-slice

We sample ~30 multi-hop questions from `hotpot_qa` (distractor split) and load their **supporting paragraphs** into our corpus. HotpotQA gives us free supervision: each question already lists which paragraphs (`title`, `sent_id`) are needed to answer it, so we can put those in the gold set automatically.

Short, factual paragraphs — a third stylistic contrast.

In [4]:
from datasets import load_dataset

HOTPOT_DIR = CACHE_DIR / "hotpot"
HOTPOT_DIR.mkdir(parents=True, exist_ok=True)
HOTPOT_CACHE = HOTPOT_DIR / "sample.json"

N_HOTPOT_QUESTIONS = 30
HOTPOT_SEED = 42


def load_hotpot() -> tuple[list[Document], list[dict]]:
    if HOTPOT_CACHE.exists():
        payload = json.loads(HOTPOT_CACHE.read_text())
    else:
        ds = load_dataset("hotpot_qa", "distractor", split="validation")
        rng = random.Random(HOTPOT_SEED)
        idxs = rng.sample(range(len(ds)), N_HOTPOT_QUESTIONS)
        rows: list[dict] = []
        for idx in idxs:
            r = ds[idx]
            rows.append({
                "id": r["id"],
                "question": r["question"],
                "answer": r["answer"],
                "level": r["level"],
                "type": r["type"],
                "supporting_facts": r["supporting_facts"],
                "context": r["context"],
            })
        payload = rows
        HOTPOT_CACHE.write_text(json.dumps(payload))

    documents: list[Document] = []
    seen: set[str] = set()
    for row in payload:
        titles = row["context"]["title"]
        sentences_lists = row["context"]["sentences"]
        for title, sents in zip(titles, sentences_lists):
            article_text = " ".join(sents).strip()
            doc_id = f"hotpot::{title}"
            if doc_id in seen or not article_text:
                continue
            seen.add(doc_id)
            documents.append(
                Document(
                    page_content=article_text,
                    metadata={
                        "source": Source.HOTPOT.value,
                        "title": title,
                        "doc_id": doc_id,
                        "url": "",
                    },
                )
            )
    return documents, payload


hotpot_docs, hotpot_rows = load_hotpot()
print(f"Loaded {len(hotpot_docs)} HotpotQA paragraphs from {len(hotpot_rows)} questions")

Loaded 300 HotpotQA paragraphs from 30 questions


## Step 4 — Chunk, embed, and persist

All three sources go through the **same** `RecursiveCharacterTextSplitter` (same chunk size and overlap) into one Chroma collection. Each chunk carries a `source` metadata field so the third notebook can do metadata filtering and the fifth notebook can break down which sources each variant retrieves from.

We also build a parallel BM25 index over the same chunks so the third notebook's hybrid retriever has both a dense and sparse view of the corpus.

### A subtle but critical detail: contextual headers

A chunk's metadata (e.g. `source="beehiiv"`, `title="(Re-) Ranking RAG Solutions"`) **never enters the embedding** — only the body text does. So if a user asks *"what does the (Re-) Ranking RAG Solutions post recommend about BM25?"*, the dense embedder has no signal that any particular chunk came from that post. The most topically-similar chunk wins, which often isn't the right one.

The fix is one line: prepend a small `[source: ... | title: ...]` header to each chunk's text **before** embedding. Now:

- The embedding encodes provenance, so title-/source-anchored queries surface the right chunks.
- BM25 catches the literal title term too (sparse retrieval gets a free upgrade).
- The LLM sees the source label inline at synthesis time — easier to cite correctly.

This is a tiny version of what's marketed as "contextual retrieval." The lesson generalizes: **embeddings can only find what's in the text. If you want something to be searchable, put it in the text.**

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def chunk_header(meta: dict) -> str:
    """Tiny provenance header injected into each chunk's text before embedding.

    This is what makes title- and source-anchored queries (e.g. "what does
    the X post say about Y?") actually retrievable. Without this, the only
    signal an embedding model has about a chunk is its body text — its
    metadata is invisible to retrieval.
    """
    title = (meta.get("title") or "").strip()
    return f"[source: {meta['source']} | title: {title}]"


all_source_docs = beehiiv_docs + wiki_docs + hotpot_docs
print(f"Chunking {len(all_source_docs)} source docs (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})...")

chunks: list[Document] = []
for d in all_source_docs:
    pieces = splitter.split_text(d.page_content)
    base = d.metadata
    header = chunk_header(base)
    for i, piece in enumerate(pieces):
        chunk_id = f"{base['doc_id']}::chunk_{i}"
        chunks.append(
            Document(
                page_content=f"{header}\n\n{piece}",
                metadata={**base, "chunk_id": chunk_id, "chunk_idx": i},
            )
        )

per_source = {}
for c in chunks:
    per_source[c.metadata["source"]] = per_source.get(c.metadata["source"], 0) + 1
print(f"Built {len(chunks)} chunks ({per_source})")
print("\nExample chunk text (note the header):\n")
print(chunks[0].page_content[:300] + " ...")

Chunking 355 source docs (size=800, overlap=120)...
Built 3689 chunks ({'beehiiv': 92, 'wikipedia': 3231, 'hotpot': 366})

Example chunk text (note the header):

[source: beehiiv | title: Evaluating AI Agent Tool Selection]

Evaluating AI Agent Tool SelectionAI Office HoursLoginSubscribe0AI Office HoursPostsEvaluating AI Agent Tool SelectionEvaluating AI Agent Tool SelectionWhen you have a hammer (in the first position), everything looks like a nailSinan Ozd ...


In [6]:
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

embeddings = get_embeddings()
print(f"Embedding {len(chunks)} chunks with MiniLM and persisting to {CHROMA_DIR}...")
chroma = build_chroma_index(
    chunks,
    persist_dir=CHROMA_DIR,
    collection_name=CHROMA_COLLECTION,
    embeddings=embeddings,
)
print(f"Chroma collection '{CHROMA_COLLECTION}' has {chroma._collection.count()} chunks")

print("Building BM25 index...")
tokenized_corpus = [c.page_content.lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)
save_bm25(bm25, chunks)
print(f"BM25 saved to data/bm25_index.pkl")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7535.82it/s]


Embedding 3689 chunks with MiniLM and persisting to /Users/sinanozdemir/Teaching/Pearson/advanced-agentic-ai-in-three-weeks/notebooks/week1/data/chroma_db...
Chroma collection 'advanced_agentic_week1' has 3689 chunks
Building BM25 index...
BM25 saved to data/bm25_index.pkl


## Step 5 — Build the gold set

Three groups, ~30 questions total:

1. **Cross-source multi-hop** (~10) — hand-curated questions that require synthesizing across two or more sources. These are where the **adaptive workflow** in the fifth notebook should dominate the simple agents.
2. **Single-source controls** (~10) — easier questions answerable from a single source. These are where the **cheap tool-calling agent** should be cost-competitive.
3. **HotpotQA-derived** (~10) — sampled from the HotpotQA set above, with `required_evidence_ids` populated automatically from the dataset's `supporting_facts`.

Each row carries `difficulty` and `hop_count` so the fifth notebook can plot quality-vs-cost broken down by complexity.

In [7]:
# 1) Hand-curated cross-source multi-hop questions (10).
#    Each one explicitly needs evidence from 2+ sources to answer well.
cross_source: list[GoldQuestion] = [
    GoldQuestion(
        id="cross_01",
        question=(
            "What does the AI Office Hours post on tool-selection evaluation say about "
            "positional bias, and how does that connect to the broader idea of "
            "supervised learning evaluation as discussed on Wikipedia?"
        ),
        reference_answer=(
            "The Beehiiv post argues that LLM agents are biased toward tools listed "
            "earlier in the prompt, so accuracy on tool-selection benchmarks is sensitive "
            "to ordering. Wikipedia's articles on machine learning and information "
            "retrieval describe how supervised evaluation depends on a held-out test set "
            "and on metrics that are robust to spurious correlations; positional bias is "
            "exactly such a spurious correlation."
        ),
        required_sources=["beehiiv", "wikipedia"],
        difficulty="hard",
        hop_count=2,
    ),
    GoldQuestion(
        id="cross_02",
        question=(
            "Compare what the AI Office Hours '(Re-) Ranking RAG Solutions' post recommends "
            "to how Wikipedia describes BM25 and cross-encoders as information retrieval "
            "techniques."
        ),
        reference_answer=(
            "The Beehiiv post advocates a two-stage pipeline: cheap first-pass retrieval "
            "(dense or sparse) followed by a cross-encoder reranker on the top candidates. "
            "Wikipedia describes BM25 as a probabilistic sparse retrieval ranking function "
            "based on term frequency and document length, and cross-encoders as bi-text "
            "models that score (query, document) pairs jointly with attention. The post's "
            "recipe layers these: BM25 for cheap recall, cross-encoder for precision."
        ),
        required_sources=["beehiiv", "wikipedia"],
        difficulty="hard",
        hop_count=2,
    ),
    GoldQuestion(
        id="cross_03",
        question=(
            "How does Sinan's argument in 'Beyond Benchmarks' relate to what Wikipedia "
            "says about the historical AI winters?"
        ),
        reference_answer=(
            "'Beyond Benchmarks' argues that benchmark scores can mask real capability "
            "gaps and that practitioners should evaluate agents on their actual tasks. "
            "Wikipedia's AI history articles describe AI winters as periods when "
            "expectations outran demonstrable real-world performance, leading to funding "
            "collapses. Both make the same underlying point: surrogate metrics that drift "
            "from real-world utility eventually correct violently."
        ),
        required_sources=["beehiiv", "wikipedia"],
        difficulty="hard",
        hop_count=2,
    ),
    GoldQuestion(
        id="cross_04",
        question=(
            "What does the 'Quantizing LLMs like LLama 3' post claim about quantization "
            "trade-offs, and how does this relate to how Wikipedia describes neural "
            "network inference?"
        ),
        reference_answer=(
            "The post argues that quantization (e.g. 4-bit or 8-bit) trades a small amount "
            "of model accuracy for large reductions in memory and latency, and that the "
            "trade-off depends on task and model size. Wikipedia describes neural network "
            "inference as a sequence of matrix multiplications whose memory and FLOPs scale "
            "with parameter count and precision, which is why lower-precision arithmetic "
            "yields the speedup the post describes."
        ),
        required_sources=["beehiiv", "wikipedia"],
        difficulty="medium",
        hop_count=2,
    ),
    GoldQuestion(
        id="cross_05",
        question=(
            "Connect the 'AIs supervising AIs' post to the Wikipedia entry on RLHF: what "
            "is the underlying technique and why does it matter?"
        ),
        reference_answer=(
            "Both describe using one model to provide feedback that trains another. The "
            "Beehiiv post focuses on practical 'AI feedback' loops where a model grades "
            "another model's outputs. Wikipedia's RLHF article formalizes this as training "
            "a reward model on human (or AI) preferences and then optimizing a policy "
            "against that reward via PPO or DPO. RLAIF (RL from AI feedback) is the "
            "specific term for replacing human labelers with an AI grader."
        ),
        required_sources=["beehiiv", "wikipedia"],
        difficulty="hard",
        hop_count=3,
    ),
    GoldQuestion(
        id="cross_06",
        question=(
            "According to Wikipedia, what role did Geoffrey Hinton play in the deep "
            "learning revival, and how does the 'Probing LLMs for a World Model' Beehiiv "
            "post connect to that intellectual lineage?"
        ),
        reference_answer=(
            "Wikipedia describes Hinton as a key figure in the 1980s revival of "
            "backpropagation and the 2012 AlexNet breakthrough that started the deep "
            "learning era. The Beehiiv post probes whether LLMs have learned implicit "
            "world models, which extends Hinton's long-running argument that distributed "
            "neural representations encode meaningful structure. The lineage runs from "
            "early connectionist representations to today's question of whether scaled "
            "transformers internalize world models."
        ),
        required_sources=["beehiiv", "wikipedia"],
        difficulty="hard",
        hop_count=2,
    ),
    GoldQuestion(
        id="cross_07",
        question=(
            "What is the relationship between BM25 (as defined on Wikipedia) and the "
            "two-stage retrieval recipe in Sinan's re-ranking post?"
        ),
        reference_answer=(
            "BM25 is a sparse, term-frequency-based retrieval scoring function, "
            "well-defined on Wikipedia. The Beehiiv re-ranking post uses BM25 (or a dense "
            "vector retriever) as the cheap first stage to fetch a candidate pool, then "
            "applies a cross-encoder reranker on those candidates. BM25 contributes recall, "
            "the cross-encoder contributes precision."
        ),
        required_sources=["beehiiv", "wikipedia"],
        difficulty="medium",
        hop_count=2,
    ),
    GoldQuestion(
        id="cross_08",
        question=(
            "Both the AI Office Hours posts and Wikipedia discuss 'evaluation' of AI "
            "systems. Synthesize a coherent view of what good evaluation looks like in "
            "2026, drawing on at least one Beehiiv post and one Wikipedia article."
        ),
        reference_answer=(
            "Good evaluation in 2026 combines (a) task-realistic test cases (the Beehiiv "
            "argument that benchmarks must reflect production tasks), (b) statistical "
            "rigor including held-out splits and metric robustness (Wikipedia's machine "
            "learning evaluation), (c) bias diagnostics such as ordering / positional bias "
            "for agents (Beehiiv tool-selection post), and (d) judging mechanisms that "
            "aren't themselves leakage-prone (RLHF / LLM-as-judge per Wikipedia + Beehiiv)."
        ),
        required_sources=["beehiiv", "wikipedia"],
        difficulty="hard",
        hop_count=3,
    ),
    GoldQuestion(
        id="cross_09",
        question=(
            "How do the historical perceptron and the modern Transformer (per Wikipedia) "
            "connect to the contemporary RAG recipe described in the AI Office Hours posts?"
        ),
        reference_answer=(
            "Wikipedia traces the perceptron (Rosenblatt) as the earliest trainable neural "
            "model, and the Transformer (Vaswani et al.) as the architecture underlying "
            "modern LLMs and embedding models. The Beehiiv RAG posts assume a Transformer "
            "embedding model for dense retrieval and a Transformer LLM for generation; "
            "without that line of architectural progress, modern RAG would not exist."
        ),
        required_sources=["beehiiv", "wikipedia"],
        difficulty="medium",
        hop_count=2,
    ),
    GoldQuestion(
        id="cross_10",
        question=(
            "Integrate the AI Office Hours discussions of agent tool-selection with "
            "Wikipedia's definition of reinforcement learning: how could RL principles "
            "improve tool selection?"
        ),
        reference_answer=(
            "Wikipedia frames reinforcement learning as learning a policy that maximizes "
            "long-run reward via interaction. The Beehiiv tool-selection post highlights "
            "that LLM agents currently choose tools via a single forward pass that is "
            "sensitive to ordering. RL principles suggest treating tool selection as a "
            "policy that is updated based on downstream task reward (or LLM-judged success), "
            "which is exactly the RLHF / RLAIF approach to aligning agent behavior."
        ),
        required_sources=["beehiiv", "wikipedia"],
        difficulty="hard",
        hop_count=3,
    ),
]
print(f"Hand-curated cross-source questions: {len(cross_source)}")

Hand-curated cross-source questions: 10


In [8]:
# 2) Single-source controls (10): easier, answerable from one source.
single_source: list[GoldQuestion] = [
    GoldQuestion(
        id="single_b1",
        question="What does Sinan's '(Re-) Ranking RAG Solutions' post recommend as a baseline reranker?",
        reference_answer=(
            "A cross-encoder model (such as a MS MARCO MiniLM cross-encoder) applied as a "
            "second stage on top of dense or sparse retrieval results."
        ),
        required_sources=["beehiiv"],
        difficulty="easy",
        hop_count=1,
    ),
    GoldQuestion(
        id="single_b2",
        question="What problem does the 'Beyond Benchmarks' post warn about?",
        reference_answer=(
            "It warns that benchmark scores can mislead practitioners because high benchmark "
            "performance does not necessarily translate to real-world task utility."
        ),
        required_sources=["beehiiv"],
        difficulty="easy",
        hop_count=1,
    ),
    GoldQuestion(
        id="single_b3",
        question="Per the AI Office Hours quantization post, when is quantization a good trade-off?",
        reference_answer=(
            "When latency or memory pressure is high and a small accuracy degradation is "
            "acceptable; particularly attractive for larger models where the absolute savings "
            "outweigh the marginal accuracy loss."
        ),
        required_sources=["beehiiv"],
        difficulty="medium",
        hop_count=1,
    ),
    GoldQuestion(
        id="single_w1",
        question="According to Wikipedia, what does BM25 stand for and what is its core scoring intuition?",
        reference_answer=(
            "BM25 stands for 'Best Matching 25.' It scores a document for a query as a sum, "
            "over query terms, of a term-frequency factor saturated by a length-normalized "
            "denominator, weighted by inverse document frequency."
        ),
        required_sources=["wikipedia"],
        difficulty="easy",
        hop_count=1,
    ),
    GoldQuestion(
        id="single_w2",
        question="Who proposed the Transformer architecture and in what paper?",
        reference_answer=(
            "Vaswani et al. proposed the Transformer in the 2017 paper 'Attention Is All "
            "You Need.'"
        ),
        required_sources=["wikipedia"],
        difficulty="easy",
        hop_count=1,
    ),
    GoldQuestion(
        id="single_w3",
        question="What was AlexNet's significance in the history of deep learning, per Wikipedia?",
        reference_answer=(
            "AlexNet won the 2012 ImageNet competition by a large margin using a deep "
            "convolutional neural network trained on GPUs, which is widely credited with "
            "starting the modern deep learning era."
        ),
        required_sources=["wikipedia"],
        difficulty="medium",
        hop_count=1,
    ),
    GoldQuestion(
        id="single_w4",
        question="What is reinforcement learning from human feedback (RLHF), per Wikipedia?",
        reference_answer=(
            "RLHF is a training approach where a reward model is fitted to human preference "
            "judgments over model outputs and then a language model policy is optimized "
            "against that reward, typically with PPO or DPO."
        ),
        required_sources=["wikipedia"],
        difficulty="medium",
        hop_count=1,
    ),
    GoldQuestion(
        id="single_w5",
        question="What is retrieval-augmented generation (RAG), per Wikipedia?",
        reference_answer=(
            "RAG is a technique that augments a language model's generation by retrieving "
            "relevant external documents at query time and conditioning the model on those "
            "retrieved documents."
        ),
        required_sources=["wikipedia"],
        difficulty="easy",
        hop_count=1,
    ),
    GoldQuestion(
        id="single_w6",
        question="When did the Dartmouth Workshop take place and why was it significant?",
        reference_answer=(
            "The Dartmouth Workshop took place in 1956 and is considered the founding "
            "event of artificial intelligence as a research field."
        ),
        required_sources=["wikipedia"],
        difficulty="easy",
        hop_count=1,
    ),
    GoldQuestion(
        id="single_w7",
        question="What is the Turing test, per Wikipedia?",
        reference_answer=(
            "The Turing test is Alan Turing's 1950 thought experiment in which a human "
            "evaluator engages in natural language conversation with an unseen interlocutor "
            "and must judge whether it is human or machine."
        ),
        required_sources=["wikipedia"],
        difficulty="easy",
        hop_count=1,
    ),
]
print(f"Single-source controls: {len(single_source)}")

Single-source controls: 10


In [9]:
# 3) HotpotQA-derived (10): sample with explicit gold supporting facts.
N_HOTPOT_GOLD = 10
hotpot_gold: list[GoldQuestion] = []
rng = random.Random(HOTPOT_SEED + 1)
sampled = rng.sample(hotpot_rows, k=min(N_HOTPOT_GOLD, len(hotpot_rows)))

for i, row in enumerate(sampled):
    sf_titles = row["supporting_facts"]["title"]
    evidence_ids = sorted({f"hotpot::{t}" for t in sf_titles})
    hop = 2 if len(set(sf_titles)) >= 2 else 1
    diff = "medium" if hop > 1 else "easy"
    if row.get("level") == "hard":
        diff = "hard"
    hotpot_gold.append(
        GoldQuestion(
            id=f"hotpot_{i:02d}",
            question=row["question"],
            reference_answer=row["answer"],
            required_sources=["hotpot"],
            required_evidence_ids=evidence_ids,
            difficulty=diff,
            hop_count=hop,
        )
    )

print(f"HotpotQA-derived: {len(hotpot_gold)}")

all_gold = cross_source + single_source + hotpot_gold
write_gold_set(all_gold)
print(f"\nTotal gold-set: {len(all_gold)} questions written to data/gold_set.jsonl")
print({d: sum(1 for q in all_gold if q.difficulty == d) for d in ["easy", "medium", "hard"]})

HotpotQA-derived: 10

Total gold-set: 30 questions written to data/gold_set.jsonl
{'easy': 7, 'medium': 6, 'hard': 17}


## Done

The corpus is built and the gold set is on disk. From here:

- The first notebook will demo what happens when you point a single-shot RAG at this messy 3-source corpus.
- Notebooks 2-4 progressively add multi-hop, hybrid + reranking, and context-window strategies.
- The fifth notebook runs the bake-off across the full gold set.

Sanity check:

In [10]:
from corpus import load_chroma, load_bm25, load_gold_set

c = load_chroma()
b = load_bm25()
g = load_gold_set()

print(f"Chroma chunks: {c._collection.count()}")
print(f"BM25 docs: {len(b['documents'])}")
print(f"Gold set: {len(g)} ({sum(1 for q in g if q.hop_count > 1)} multi-hop)")

print("\nQuick dense-retrieval smoke test:")
hits = c.similarity_search("what is BM25?", k=3)
for h in hits:
    print(f"  - [{h.metadata['source']}] {h.metadata['title'][:60]}: {h.page_content[:80]}...")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5307.94it/s]


Chroma chunks: 3689
BM25 docs: 3689
Gold set: 30 (20 multi-hop)

Quick dense-retrieval smoke test:
  - [wikipedia] Okapi BM25: [source: wikipedia | title: Okapi BM25]

BM25+ is an extension of BM25. BM25+ wa...
  - [wikipedia] Okapi BM25: [source: wikipedia | title: Okapi BM25]

== Modifications ==
At the extreme valu...
  - [wikipedia] Okapi BM25: [source: wikipedia | title: Okapi BM25]

In information retrieval, Okapi BM25 (B...
